# Histopathology Model — WSI → ResNet50 Features → OS_STATUS, PFS_STATUS, Stage

Scaffold notebook, per `claude plans/as-of-now-we-refactored-tower.md` Phase 4.

**Phase 4 Objective**: develop, train, validate, and evaluate the Histopathology
prediction model as a standalone modality before integrating it into the
four-modality multimodal framework (Phase 5).

Pipeline: Whole Slide Images -> Tissue Detection / Tiling -> Stain Normalization ->
frozen ResNet50 tile embeddings -> mean-pooled per-patient feature vector ->
RandomForest / LogisticRegression (same Level-0 paradigm as Expression/Mutation) ->
feature-store save, matching the existing `{modality}/{target}/` convention.

Depends on `EXP/histopathology_data_acquisition.ipynb` having produced
`Data/raw_data/histopathology/slide_manifest.csv`. Requires
`EXP/requirements-histopathology.txt` installed (openslide-python, torch,
torchvision, histolab, opencv-python, Pillow).

In [ ]:
import os
import json
import sys
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score

# Repo root on path for histopathology_utils
sys.path.insert(0, str(Path.cwd()))
from histopathology_utils import (
    aggregate_patient_features,
    build_resnet50_encoder,
    cache_slide_embeddings,
    embed_tiles,
    tile_slide,
)

SLIDE_MANIFEST_PATH = "../Data/raw_data/histopathology/slide_manifest.csv"
TILE_CACHE_DIR = "../Data/feature_store/histopathology/tile_embeddings_cache"
FEATURE_STORE_ROOT = "../Data/feature_store/histopathology"
CLINICAL_PATH = "../Data/processed_data/mutation_data_processed/selected_clinical.csv"

os.makedirs(TILE_CACHE_DIR, exist_ok=True)

## Step 1 — WSI Preprocessing: Tissue Detection + Tiling

- Tissue detection: Otsu thresholding on a downsampled thumbnail; discard tiles
  >80-90% background.
- Tiling: 256x256px tiles at 20x magnification.
- Stain normalization: Macenko, to correct H&E batch variation across TCGA's many
  contributing sites.

Requires `openslide-python` + `histolab` (see `requirements-histopathology.txt`).
Left as a TODO pipeline stub until real slide data is downloaded (Phase 4.1).

In [ ]:
# Implemented in EXP/histopathology_utils.py — re-exported here for notebook clarity.
# tile_slide(svs_path) -> list of (tile_id, PIL.Image) with Otsu mask + Macenko norm

## Step 2 — Frozen ResNet50 Tile Embeddings

ImageNet-pretrained ResNet50 (torchvision), frozen, penultimate-layer embedding
(~2048-D), batched inference on GPU (MPS on Apple Silicon). No fine-tuning — this
keeps the pipeline consistent with the project's existing scikit-learn-only
modeling pattern (fixed feature vector -> RF/LR), per plan Phase 4.3.

In [ ]:
# build_resnet50_encoder() -> (frozen ResNet50 module, device)
# embed_tiles(encoder, device, tiles) -> (n_tiles, 2048) ndarray
# cache_slide_embeddings(slide_id, svs_path, TILE_CACHE_DIR, encoder, device) -> cache path

## Step 3 — Per-Patient Feature Aggregation

Mean-pooling is the baseline (not attention-MIL/CLAM, which is future work only,
per plan Phase 4.3): tile embeddings -> mean per slide -> mean across slides for
patients with multiple diagnostic slides.

**No PCA by default** — the baseline pipeline is ResNet50 -> 2048-D per-patient
embedding -> RandomForest / LogisticRegression directly. Only introduce PCA if the
feature count relative to the patient count causes overfitting/runtime issues in
practice.

In [ ]:
# aggregate_patient_features(slide_manifest, TILE_CACHE_DIR) implemented in histopathology_utils.py

def run_embedding_pipeline(slide_manifest: pd.DataFrame) -> None:
    """Tile + embed all slides in manifest; cache per-slide .npy embeddings."""
    encoder, device = build_resnet50_encoder()
    for i, row in slide_manifest.iterrows():
        slide_id = Path(row["slide_id"]).stem
        print(f"[{i + 1}/{len(slide_manifest)}] {row['slide_id']}")
        cache_slide_embeddings(slide_id, row["path"], TILE_CACHE_DIR, encoder, device)


# slide_manifest = pd.read_csv(SLIDE_MANIFEST_PATH)
# run_embedding_pipeline(slide_manifest)
# features_all = aggregate_patient_features(slide_manifest, TILE_CACHE_DIR)

## Step 4 — Merge with Clinical Labels, Train/Test Split

Same convention as `gene_expression_v2.ipynb` / `new_mutation_data.ipynb`: one
80/20 stratified split (stratify on OS_STATUS, random_state=42) reused across all
three targets.

In [ ]:
def load_clinical():
    clinical = pd.read_csv(CLINICAL_PATH, index_col="PATIENT_ID")
    clinical["Stage"] = clinical["AJCC_PATHOLOGIC_TUMOR_STAGE"].apply(
        lambda x: int(x) if pd.notna(x) else np.nan
    )
    clinical["OS_STATUS"] = clinical["OS_STATUS"].astype(int)
    clinical["PFS_STATUS"] = clinical["PFS_STATUS"].astype(int)
    return clinical


# clinical = load_clinical()
# ml = features_all.join(clinical, how="inner").dropna(subset=["OS_STATUS", "PFS_STATUS", "Stage"])
# embed_cols = [c for c in features_all.columns]
#
# train_idx, test_idx = train_test_split(
#     ml.index, test_size=0.2, stratify=ml["OS_STATUS"], random_state=42
# )
# X_all, X_train, X_test = ml[embed_cols], ml.loc[train_idx, embed_cols], ml.loc[test_idx, embed_cols]

## Step 5 — Train + Evaluate (RF / LR), Per Target

Same model family and CV discipline as Expression/Mutation: `RandomForestClassifier`
+ `LogisticRegression`, `StratifiedKFold(5)` CV on train only, honest AUC on the
locked-away test set.

In [ ]:
def train_and_eval(X_train, y_train, X_test, y_test, label, scoring="roc_auc"):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    models = {
        "Random Forest": RandomForestClassifier(
            n_estimators=300, random_state=42, class_weight="balanced", n_jobs=-1
        ),
        "Logistic (LASSO)": LogisticRegression(
            solver="saga", l1_ratio=1, C=0.1, random_state=42, class_weight="balanced", max_iter=5000
        ),
    }
    print(f"\n--- {label} ---")
    res = {}
    for name, model in models.items():
        cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring=scoring)
        model.fit(X_train, y_train)
        if scoring == "roc_auc_ovr_weighted":
            y_prob = model.predict_proba(X_test)
            test_auc = roc_auc_score(y_test, y_prob, multi_class="ovr", average="weighted")
        else:
            y_prob = model.predict_proba(X_test)[:, 1]
            test_auc = roc_auc_score(y_test, y_prob)
        res[name] = {"cv_scores": cv_scores, "test_auc": test_auc, "model": model}
        print(f"  {name:<25} {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}   test={test_auc:.3f}")
    return res


# os_results    = train_and_eval(X_train, ml.loc[train_idx, "OS_STATUS"],  X_test, ml.loc[test_idx, "OS_STATUS"],  "OS_STATUS — histopathology")
# pfs_results   = train_and_eval(X_train, ml.loc[train_idx, "PFS_STATUS"], X_test, ml.loc[test_idx, "PFS_STATUS"], "PFS_STATUS — histopathology")
# stage_results = train_and_eval(X_train, ml.loc[train_idx, "Stage"],      X_test, ml.loc[test_idx, "Stage"],      "Stage — histopathology", scoring="roc_auc_ovr_weighted")

## Step 6 — Save to Feature Store

Matches the existing convention exactly:
```
Data/feature_store/histopathology/
  OS_STATUS/{features.csv, metadata.json, model.joblib, test_predictions.csv}
  PFS_STATUS/  ...
  Stage/       ...
registry_histopathology.json
```
`test_predictions.csv` is written from the start here (Expression/Mutation had to
be backfilled retroactively — see `EXP/backfill_test_predictions.py` — Histopathology
establishes the contract immediately so Phase 6 needs no backfill for this modality).

In [ ]:
def save_target_to_feature_store(target, model_name, model, X_all_target, y_all, X_test, y_test, meta_extra):
    store_dir = os.path.join(FEATURE_STORE_ROOT, target)
    os.makedirs(store_dir, exist_ok=True)

    X_all_target.to_csv(os.path.join(store_dir, "features.csv"))
    joblib.dump(model, os.path.join(store_dir, "model.joblib"))

    proba = model.predict_proba(X_test)
    pred = model.predict(X_test)
    if target == "Stage":
        out = pd.DataFrame({"patient_id": X_test.index, "true_label": y_test.values, "predicted_label": pred})
        for i, cls in enumerate(model.classes_):
            out[f"probability_class_{int(cls)}"] = proba[:, i]
    else:
        out = pd.DataFrame(
            {
                "patient_id": X_test.index,
                "true_label": y_test.values,
                "predicted_label": pred,
                "predicted_probability": proba[:, 1],
            }
        )
    out.to_csv(os.path.join(store_dir, "test_predictions.csv"), index=False)

    meta = {
        "target": target,
        "model_type": model_name,
        "n_features": X_all_target.shape[1],
        "created": str(date.today()),
        **meta_extra,
    }
    with open(os.path.join(store_dir, "metadata.json"), "w") as f:
        json.dump(meta, f, indent=2)
    return meta


# registry = {}
# for target, results in {"OS_STATUS": os_results, "PFS_STATUS": pfs_results, "Stage": stage_results}.items():
#     best_name = max(results, key=lambda k: results[k]["test_auc"])
#     best = results[best_name]
#     meta = save_target_to_feature_store(
#         target, best_name, best["model"], X_all, ml[target], X_test, ml.loc[test_idx, target],
#         meta_extra={
#             "cv_auc_mean": round(float(best["cv_scores"].mean()), 4),
#             "cv_auc_std": round(float(best["cv_scores"].std()), 4),
#             "test_auc": round(float(best["test_auc"]), 4),
#             "train_patients": len(train_idx),
#             "test_patients": len(test_idx),
#             "total_patients": len(ml),
#         },
#     )
#     registry[target] = meta
#
# with open(os.path.join(FEATURE_STORE_ROOT, "../registry_histopathology.json"), "w") as f:
#     json.dump(registry, f, indent=2)